# 🧬 ADN : Single Model Theory Optimization
**Objectif** : Identifier le modèle unique théoriquement optimal pour le dataset de conversion (Bernoulli/Poisson monotone).
**Contraintes** : Pas d'ensemble, pas de règles, pas de target encoding. Juste XGBoost + Physique des Données (Monotonie).

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import itertools

SEED = 42
print("Setup Complete.")

In [ ]:
# Loading
df_train = pd.read_csv('conversion_data_train.csv')
features = ['country', 'source', 'new_user', 'age', 'total_pages_visited']

# Minimal Preprocessing (Label Encoding)
for col in ['country', 'source']:
    le = LabelEncoder()
    df_train[col] = le.fit_transform(df_train[col])

X = df_train[features]
y = df_train['converted']
print(f"Data Loaded: {X.shape}")

In [ ]:
# Theory: Bernoulli MLE + Monotonicity
# Age decreases (-1), Pages increases (1)
feature_monotony = '(0, 0, 0, -1, 1)'

print(f"Monotonic Constraints: {feature_monotony}")

In [ ]:
# Hyperparameters
param_grid = {
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [3, 4, 5, 6],
    'n_estimators': [300, 500, 800],
    'subsample': [0.8, 1.0]
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Testing {len(combinations)} combinations...")

best_score = 0
best_params = {}

for i, params in enumerate(combinations):
    cv_f1 = []

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(**params, 
                                  objective='binary:logistic', 
                                  n_jobs=-1, 
                                  random_state=SEED,
                                  monotone_constraints=feature_monotony,
                                  eval_metric='logloss')
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]

        # Optimize Threshold
        best_f1_fold = 0
        for th in np.arange(0.3, 0.6, 0.02):
            f = f1_score(y_val, (probs >= th).astype(int))
            if f > best_f1_fold: best_f1_fold = f

        cv_f1.append(best_f1_fold)

    mean_f1 = np.mean(cv_f1)

    if mean_f1 > best_score:
        best_score = mean_f1
        best_params = params
        print(f"🌟 New Best: F1={mean_f1:.5f} | Params={params}")

print("Done.")

In [ ]:
print("="*80)
print("🏆 CONFIGURATION FINALE (ADN DU DATASET)")
print(f"Params: {best_params}")
print(f"Avg F1 (5-Fold): {best_score:.5f}")